# 🔬 SurfaceGuard AI — Visual Error Analysis
### Detailed Validation Breakdown: Missed Defects, False Alarms & Class Confusion

This notebook performs a deep-dive analysis on the model's predictions on the **20% Stratified Validation Set** (`2,278 images × 4 defect classes = 9,112 evaluation pairs`).

---
### Objectives:
1. **Per-Class Performance Audit**: Calculate exact Foreground Dice scores for `defect_a`, `defect_b`, `defect_c`, and `defect_d`.
2. **False Negative Analysis**: Visually inspect the largest defects missed by the model.
3. **False Positive Analysis**: Inspect what clean steel features (glare, shadows, roller marks) caused false alarms.
4. **Class Confusion Audit**: Determine if defects are being detected under the wrong category.
5. **Threshold & Area Optimization**: Sweep decision thresholds to find the highest-scoring parameters.


In [ ]:
import os
import sys

# Ensure repository root is in sys.path
sys.path.append(os.path.abspath('..'))

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from src.evaluate import evaluate_validation_set, visualize_defect_comparison
from src.dataset import DEFECT_CLASSES
from src.utils import rle_to_mask, mask_to_rle

# Set clean aesthetic styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.dpi'] = 120
print("Environment initialized successfully!")


---
## 1. Load or Generate Validation Predictions CSV
If `val_predictions.csv` has not been generated yet, `evaluate_validation_set()` will run inference across the holdout set and export the full comparison.


In [ ]:
VAL_CSV_PATH = '../val_predictions.csv'
DATA_CSV = '../data/train.csv'
IMAGE_DIR = '../data/train'
CHECKPOINT_PATH = '../models/best_scratch_unet.pth'

if os.path.exists(VAL_CSV_PATH):
    print(f"Loading existing validation predictions from {VAL_CSV_PATH}...")
    val_df = pd.read_csv(VAL_CSV_PATH)
else:
    print(f"Generating validation predictions using checkpoint: {CHECKPOINT_PATH}...")
    val_df, metrics = evaluate_validation_set(
        data_csv=DATA_CSV,
        image_dir=IMAGE_DIR,
        checkpoint_path=CHECKPOINT_PATH,
        output_csv=VAL_CSV_PATH,
        batch_size=8,
        threshold=0.50
    )

print(f"Total evaluation rows: {len(val_df):,}")
display(val_df.head(10))


---
## 2. Metric Breakdown & Class Performance Dashboard
Let's analyze the performance distribution across all 4 defect classes.


In [ ]:
# 1. Error Type Counts per Class
error_counts = pd.crosstab(val_df['defect_type'], val_df['error_type'])
print("=== Error Type Breakdown per Defect Class ===")
display(error_counts)

# 2. Per-Class Foreground Dice (evaluated strictly on positive ground-truth samples)
pos_val_df = val_df[val_df['gt_pixels'] > 0].copy()
per_class_dice = pos_val_df.groupby('defect_type')['dice'].mean()
print("
=== Per-Class Mean Foreground Dice ===")
for cls_name, score in per_class_dice.items():
    print(f"  {cls_name:<10}: F1 = {score:.4f}")

# Plotting Dashboard
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot A: Foreground Dice per Class
sns.barplot(x=per_class_dice.index, y=per_class_dice.values, ax=axes[0], palette="viridis")
axes[0].set_title("Mean Foreground Dice per Defect Class (GT > 0)", fontsize=13, fontweight='bold')
axes[0].set_ylabel("Dice Score")
axes[0].set_ylim(0, 1.0)
for p in axes[0].patches:
    axes[0].annotate(f"{p.get_height():.3f}", (p.get_x() + p.get_width() / 2., p.get_height() + 0.02),
                     ha='center', fontsize=11, fontweight='bold')

# Plot B: Error Distribution
error_counts.plot(kind='bar', stacked=True, ax=axes[1], colormap="Set2")
axes[1].set_title("Prediction Outcomes per Class (TP / FP / FN / TN)", fontsize=13, fontweight='bold')
axes[1].set_ylabel("Image Count")
axes[1].legend(title="Outcome", bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()


---
## 3. Visual Error Analysis: Worst False Negatives (Missed Defects)
False Negatives occur when a defect was present in the ground truth (`gt_pixels > 0`), but the model predicted nothing or completely missed it (`dice == 0`).
Let's inspect the largest missed defects to understand **why** the model missed them.


In [ ]:
fn_df = val_df[val_df['error_type'] == 'FN'].sort_values(by='gt_pixels', ascending=False)
print(f"Total False Negatives (Missed Defects): {len(fn_df)}")
display(fn_df[['image_id', 'defect_type', 'gt_pixels', 'max_prob']].head(10))

# Display top 3 largest missed defects
top_fns = fn_df.head(3)
for idx, row in top_fns.iterrows():
    visualize_defect_comparison(
        image_dir=IMAGE_DIR,
        image_id=row['image_id'],
        defect_type=row['defect_type'],
        gt_rle=row['gt_rle'],
        pred_rle=row['pred_rle'],
        dice=row['dice'],
        max_prob=row['max_prob'],
        error_type=row['error_type']
    )


---
## 4. Visual Error Analysis: Worst False Positives (Hallucinations on Clean Steel)
False Positives occur when steel has no defect (`gt_pixels == 0`), but the model hallucinated a defect.
Let's see what visual artifacts (glare, reflections, shadows, seams) confused the model.


In [ ]:
fp_df = val_df[val_df['error_type'] == 'FP'].sort_values(by='pred_pixels', ascending=False)
print(f"Total False Positives (Hallucinations): {len(fp_df)}")
display(fp_df[['image_id', 'defect_type', 'pred_pixels', 'max_prob']].head(10))

# Display top 3 largest false alarms
top_fps = fp_df.head(3)
for idx, row in top_fps.iterrows():
    visualize_defect_comparison(
        image_dir=IMAGE_DIR,
        image_id=row['image_id'],
        defect_type=row['defect_type'],
        gt_rle=row['gt_rle'],
        pred_rle=row['pred_rle'],
        dice=row['dice'],
        max_prob=row['max_prob'],
        error_type=row['error_type']
    )


---
## 5. Visual Inspection: Best True Positives (High Dice Matches)
Let's inspect cases where the model succeeded with high accuracy (`dice > 0.70`).


In [ ]:
tp_df = val_df[val_df['error_type'] == 'TP'].sort_values(by='dice', ascending=False)
print(f"Total Strong True Positives (Dice >= 0.50): {len(tp_df)}")
display(tp_df[['image_id', 'defect_type', 'gt_pixels', 'pred_pixels', 'dice']].head(10))

# Display top 3 best predictions
top_tps = tp_df.head(3)
for idx, row in top_tps.iterrows():
    visualize_defect_comparison(
        image_dir=IMAGE_DIR,
        image_id=row['image_id'],
        defect_type=row['defect_type'],
        gt_rle=row['gt_rle'],
        pred_rle=row['pred_rle'],
        dice=row['dice'],
        max_prob=row['max_prob'],
        error_type=row['error_type']
    )


---
## 6. Probability Distribution Analysis
Let's examine the peak probability distributions for each class when a defect is actually present versus when the steel is clean.
This reveals the exact threshold boundary separating true defects from background noise.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for idx, defect_name in enumerate(DEFECT_CLASSES):
    subset = val_df[val_df['defect_type'] == defect_name]
    pos = subset[subset['gt_pixels'] > 0]['max_prob']
    neg = subset[subset['gt_pixels'] == 0]['max_prob']

    sns.kdeplot(pos, ax=axes[idx], label='Defect Present (GT > 0)', color='green', fill=True, common_norm=False)
    sns.kdeplot(neg, ax=axes[idx], label='Clean Steel (GT = 0)', color='red', fill=True, common_norm=False)

    axes[idx].set_title(f"Peak Prob Distribution: {defect_name}", fontsize=12, fontweight='bold')
    axes[idx].set_xlabel("Max Predicted Probability")
    axes[idx].set_xlim(0, 1.0)
    axes[idx].legend(loc='upper right')

plt.tight_layout()
plt.show()


---
## 7. Key Findings & Next Actionable Steps
Summarize findings from the visual error analysis:
1. **Defect A & B Status**: Check whether probability peaks for A & B are separating from clean steel.
2. **Defect C & D Quality**: Verify if boundary errors or false alarms are dragging down F1.
3. **Threshold Recommendations**: Identify whether adjusting thresholds per-class yields higher official scores.
